# Environment setup

**Destructive.** Drops and recreates `<catalog_prefix>_<env>` and the ingest
schema, then optionally seeds a small sample source so the rest of the template
has something to read.

Run it by hand, never on a schedule:

```
databricks bundle run setup_job --target dev
```

In [0]:
import random
from datetime import datetime, timedelta

from pyspark.sql import types as T

In [0]:
# Job parameters. Every notebook in this repo reads its config the same way, so
# nothing below is workspace specific and nothing hardcodes a catalog name.
configs = dict(dbutils.notebook.entry_point.getCurrentBindings())

ENV = configs.get("env", "dev")
CATALOG_PREFIX = configs.get("catalog_prefix", "rearc")
SEED_SAMPLE_DATA = configs.get("seed_sample_data", "True").lower() == "true"

CATALOG = f"{CATALOG_PREFIX}_{ENV}"
INGEST_CATALOG = f"{CATALOG_PREFIX}_ingest"
INGEST_SCHEMA = ENV
LANDING_VOLUME = "landing"
LANDING_BASE = f"/Volumes/{INGEST_CATALOG}/{INGEST_SCHEMA}/{LANDING_VOLUME}"

SCHEMAS = ["bronze", "silver", "gold"]

print(f"ENV={ENV} | catalog={CATALOG} | ingest={INGEST_CATALOG}.{INGEST_SCHEMA} | landing={LANDING_BASE}")

## Reset

Everything below this point is safe to re-run; this cell is not.

In [0]:
spark.sql(f"DROP CATALOG IF EXISTS {CATALOG} CASCADE")
spark.sql(f"DROP SCHEMA IF EXISTS {INGEST_CATALOG}.{INGEST_SCHEMA} CASCADE")
print(f"dropped {CATALOG} and {INGEST_CATALOG}.{INGEST_SCHEMA}")

## Create catalogs, schemas, volumes

In [0]:
for catalog in [CATALOG, INGEST_CATALOG]:
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")

# One checkpoints volume per medallion schema: streaming checkpoints live in
# Unity Catalog volumes, never on DBFS.
for schema in SCHEMAS:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{schema}.checkpoints")

# The ingest catalog stands in for upstream systems: one schema per environment,
# with a landing volume for file feeds.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {INGEST_CATALOG}.{INGEST_SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {INGEST_CATALOG}.{INGEST_SCHEMA}.{LANDING_VOLUME}")

display(spark.sql(f"SHOW SCHEMAS IN {CATALOG}"))

## Sample source data

Placeholder so `bundle run bronze_ingestion_job` works end to end on a fresh
workspace. Two sources, one per ingestion pattern in `src/ingestion/`:

| Source | Delivered as | Ingested by |
|---|---|---|
| `landing/events` | JSON files on a volume | Auto Loader |
| `<ingest>.<env>.items` | Delta table, change feed on | Delta CDF stream |

Delete this section once the project has a real source.

In [0]:
if SEED_SAMPLE_DATA:
    random.seed(42)
    BASE_DATE = datetime(2025, 1, 1)

    categories = ["hardware", "software", "services", "support"]
    item_rows = [
        (
            f"ITEM-{i:04d}",
            f"Sample item {i}",
            random.choice(categories),
            round(random.uniform(5.0, 500.0), 2),
            BASE_DATE,
        )
        for i in range(1, 51)
    ]
    item_schema = T.StructType([
        T.StructField("item_id", T.StringType()),
        T.StructField("item_name", T.StringType()),
        T.StructField("category", T.StringType()),
        T.StructField("unit_price", T.DoubleType()),
        T.StructField("last_updated", T.TimestampType()),
    ])

    (
        spark.createDataFrame(item_rows, item_schema)
        .write.mode("overwrite")
        .option("delta.enableChangeDataFeed", "true")
        .saveAsTable(f"{INGEST_CATALOG}.{INGEST_SCHEMA}.items")
    )
    print(f"seeded {len(item_rows)} items (Delta, CDF on)")

In [0]:
if SEED_SAMPLE_DATA:
    event_types = ["view", "add_to_cart", "purchase", "return"]
    event_rows = []
    for i in range(1, 20001):
        item = f"ITEM-{random.randint(1, 50):04d}"
        qty = random.randint(1, 5)
        ts = BASE_DATE + timedelta(
            days=random.randint(0, 89), hours=random.randint(0, 23), minutes=random.randint(0, 59)
        )
        amount = round(random.uniform(5.0, 500.0) * qty, 2)
        # ~2% missing amounts: a data-quality wrinkle for the silver expectations
        event_rows.append(
            (f"EVT-{i:07d}", item, random.choice(event_types), qty, None if random.random() < 0.02 else amount, ts)
        )

    event_schema = T.StructType([
        T.StructField("event_id", T.StringType()),
        T.StructField("item_id", T.StringType()),
        T.StructField("event_type", T.StringType()),
        T.StructField("quantity", T.IntegerType()),
        T.StructField("amount", T.DoubleType(), True),
        T.StructField("event_timestamp", T.TimestampType()),
    ])

    (
        spark.createDataFrame(event_rows, event_schema)
        .repartition(4)
        .write.mode("overwrite")
        .json(f"{LANDING_BASE}/events")
    )
    print(f"seeded {len(event_rows)} events -> {LANDING_BASE}/events")

In [0]:
print("=" * 60)
print(f"{CATALOG} ready")
print("=" * 60)
for schema in SCHEMAS:
    print(f"  {CATALOG}.{schema}")
if SEED_SAMPLE_DATA:
    print(f"  {INGEST_CATALOG}.{INGEST_SCHEMA}.items: {spark.table(f'{INGEST_CATALOG}.{INGEST_SCHEMA}.items').count():,} rows")
    print(f"  {LANDING_BASE}/events: {spark.read.json(f'{LANDING_BASE}/events').count():,} rows")
print()
print("Next: databricks bundle run bronze_ingestion_job --target <env>")